# ಪಾಠ 13 - ಏಜೆಂಟ್ ಸ್ಮರಣೆ


## ಸೆಟಪ್

ಈ ನೋಟ್‌ಬುಕ್ **Microsoft Agent Framework** (MAF) ಬಳಸಿ **ಸ್ಥಿರ ಸ್ಮರಣೆ** ಹೊಂದಿರುವ ಪ್ರವಾಸ ಬುಕ್ಕಿಂಗ್ ಏಜೆಂಟ್ ಅನ್ನು ನಿರ್ಮಿಸುವುದನ್ನು ತೋರಿಸುತ್ತದೆ.

ನೀವು ತಿಳಿಯಲಿದ್ದೀರಿ ಏಜೆಂಟ್ ಸ್ಮರಣೆ ವಿವಿಧ ರೀತಿಗಳು — ಕಾರ್ಯ ನಿರ್ವಹಿಸುವ, ಸಂಕ್ಷಿಪ್ತ ಅವಧಿ ಮತ್ತು ದೀರ್ಘಾವಧಿ — ಏಜೆಂಟ್ ಸಂವಾದಗಳಲ್ಲಿ ಮಾಹಿತಿ ಮನಕಟ್ಟುಮಾಡಿಕೊಳ್ಳುವ ಮತ್ತು ಬಳಸುವ ರೀತಿಯನ್ನು ಹೇಗೆ ಪ್ರಭಾವಿಸುತ್ತದೆ.

**ಆವಶ್ಯಕತೆಗಳು:**
- ಚಟುವಟಿಕೆಯ ಜೊತೆ ಡಿಪ್ಲಾಯ್ಡ್ ಚಾಟ್ ಮಾದರಿಯನ್ನು ಹೊಂದಿರುವ Microsoft Foundry ಪ್ರಾಜೆಕ್ಟ್ (ಉದಾ. `gpt-4.1-mini`).
- ಅಜರ್ CLI ಯೊಂದಿಗೆ ಲಾಗಿನ್ ಆಗಿರುವಿರಿ — ನಿಮ್ಮ ಟರ್ಮಿನಲ್‌ನಲ್ಲಿ `az login` ಚಲಾಯಿಸಿ.
- `AZURE_AI_PROJECT_ENDPOINT` — ನಿಮ್ಮ Microsoft Foundry ಪ್ರಾಜೆಕ್ಟ್ ಎಂಡ್‌ಪಾಯಿಂಟ್.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ನಿಮ್ಮ ಡಿಪ್ಲಾಯ್ಡ್ ಮಾದರಿಯ ಹೆಸರು.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ಏಜೆಂಟ್ ಮೆಮೊರಿಯ ಪ್ರಕಾರಗಳು

AI ಏಜೆಂಟ್ಗಳು ವಿಭಿನ್ನ ಪ್ರಕಾರಗಳ ಮೆಮೊರಿಯನ್ನು ಉಪಯೋಗಿಸಬಹುದು, ಪ್ರತಿಯೊಬ್ಬವು ವಿಭಿನ್ನ ಉದ್ದೇಶವನ್ನೂ ಸೇವಿಸುತ್ತದೆ:

### ಕೆಲಸ ಮಾಡುವ ಮೆಮೊರಿ
ಸಂಭಾಷಣಾ ಥ್ರೆಡ್ ಸ್ವತಃ — ಒಂದೇ ಸೆಷನ್‌ನಲ್ಲಿ ವಿನಿಮಯವಾಗುವ ಸಂದೇಶಗಳು. ಏಜೆಂಟ್ coherence ಕಾಳಜಿ ಕಾಪಾಡಲು ಅದೇ ಥ್ರೆಡ್‌ನ ಹಿಂದಿನ ಸಂದೇಶಗಳನ್ನು ಹಿಂದಿರುಗಿ ನೋಡಬಹುದು. MAF ನಲ್ಲಿ ಇದು **`agent.create_session()`** ಮೂಲಕ ರಚನೆಯಾಗುತ್ತದೆ, ಇದು `AgentSession` ಅನ್ನು ಹಿಂತಿರುಗಿಸುತ್ತದೆ.

### ಚಿಕ್ಕಕಾಲಿಕ ಮೆಮೊರಿ
ಒಂದು ಕಾರ್ಯ ಅಥವಾ ಸೆಷನ್ ಅವಧಿಗೆ ಉಳಿಯುವ ಮಾಹಿತಿ ಆದರೆ ಶಾಶ್ವತವಾಗಿ ಸಂಗ್ರಹಿಸಲಾಗುವುದಿಲ್ಲ. ಉದಾಹರಣೆಗೆ, ಏಜೆಂಟ್ ಬಹು-ಟರ್ನ್ ಯೋಜನೆ ಸಂಭಾಷಣೆಯ ಸಮಯದಲ್ಲಿ ವಾಸ್ತವಗಳನ್ನು ಸಂಗ್ರಹಿಸಿ ಅಂತಿಮ ಪಥಯೋಜನೆ ಸೃಷ್ಟಿಸಲು ಬಳಸಬಹುದು.

### ದೀರ್ಘಕಾಲಿಕ ಮೆಮೊರಿ
ಸೆಷನ್‌ಗಳು **ಮೂಲಕ** ಉಳಿಯುವ ಇಚ್ಛೆಗಳು ಮತ್ತು ವಾಸ್ತವಗಳು. ಮರಳಿದ ಬಳಕೆದಾರರು ತಮ್ಮ ಆಹಾರದ ನಿಯಮಗಳು ಅಥವಾ ಪ್ರಯಾಣ ಶೈಲಿಯನ್ನು ಮತ್ತೆ ಹೇಳಬೇಕಾಗಿಲ್ಲ. ದೀರ್ಘಕಾಲಿಕ ಮೆಮೊರಿ ಸಾಮಾನ್ಯವಾಗಿ ಹೊರಗಿನ ಸಂಗ್ರಹಣೆಯಿಂದ ಬೆಂಬಲಿತವಾಗಿರುತ್ತದೆ — ಡೇಟಾಬೇಸ್, ಫೈಲ್, ಅಥವಾ ವೆಕ್ಟರ್ ಸೂಚ್ಯಂಕ, ಮತ್ತು ಉಪಕರಣಗಳ ಮೂಲಕ ಏಜೆಂಟ್‌ಗೆ ತೋರಿಸಲಾಗುತ್ತದೆ.


## ಸೆಷನ್‌ಗಳೊಂದಿಗೆ ಕಾರ್ಯಾಚರಣಾ ಮೆಮರಿ

ಮೆಮರಿಯ ಸರಳ ರೂಪವೇ ಸಂಭಾಷಣಾ ಸೆಷನ್. ನೀವು ಒಂದೇ ಸೆಷನ್ ಅನ್ನು (`agent.create_session()` ಮೂಲಕ ರಚಿಸಲಾದ) ಮುಂದುವರಿದ `agent.run()` ಕರೆಗಳಿಗೆ ಸಲ್ಲಿಸಿದಾಗ, ಏಜೆಂಟ್ ಆ ಸಂಭಾಷಣೆಯ ಸಂಪೂರ್ಣ ಇತಿಹಾಸವನ್ನು ನೋಡುತ್ತದೆ ಮತ್ತು ಮುಂಚಿನ ಮಾಹಿತಿಗಳನ್ನು ಸ್ಮರಿಸಬಹುದು.

ನಾವು ಒಂದು ಪ್ರಯಾಣ ಏಜೆಂಟ್ ರಚಿಸಿ ಕಾರ್ಯಾಚರಣಾ ಮೆಮರಿಯನ್ನು ಪ್ರದರ್ಶಿಸೋಣ.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

ಏಜೆಂಟ್ ಬಜೆಟ್ ಅನ್ನು ಸರಿಯಾಗಿ ಮರೆತಿದೆ ಏಕೆಂದರೆ ಇಬ್ಬರು ಸಂದೇಶಗಳೂ ಅದೇ ಸೆಶನ್ ಅನ್ನು ಹಂಚಿಕೊಳ್ಳುತ್ತವೆ. ಇದು **ಕಾರ್ಯಕ್ಷಮತೆಯ ನೆನಪು** — ಅದು ಸೆಶನ್ ಜೀವನಾವಧಿಗೆ ಮಾತ್ರ ಇರುತ್ತದೆ.

### ಹೊಸ ಥ್ರೆಡ್‌ನಲ್ಲಿ ಏನು ಸಂಭವಿಸುತ್ತದೆ?

ನಾವು **ಹೊಸ** ಸೆಶನ್ ರಚಿಸಿದರೆ, ಏಜೆಂಟ್ ಕಳೆಯುವ ಸಂಭಾಷಣೆಯ ನೆನಪು ಹೊಂದಿಲ್ಲ:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## ದೀರ್ಘಕಾಲಿನ ಸ್ಮರಣೆ ಮಾದರಿ

ಬಳಕೆದಾರರ ಆದ್ಯತೆಗಳನ್ನು **ಸೆಷನ್‌ಗಳ ಮೂಲಕ** ನೆನಪು ಮಾಡಿಕೊಳ್ಳಲು, ಸಂವಾದದ ಥ್ರೆಡ್ ಹೊರಗಿನ ಸ್ಥಿರ ಸಂಗ್ರಹಣೆಯೊಂದನ್ನು ಬೇಕಾಗುತ್ತದೆ. ಏಜೆಂಟ್ ಈ ಸಂಗ್ರಹಣೆಯನ್ನು **ಉಪಕರಣಗಳ** ಮೂಲಕ ಪ್ರವೇಶಿಸುತ್ತದೆ — ಮಾಹಿತಿ ಉಳಿಸಲು ಮತ್ತು ಹಿಂತೆಗೆದುಕೊಳ್ಳಲು ಕರೆಯಬಹುದಾದ ಫಂಕ್ಷನುಗಳು.

ಕೆಳಗೆ ನಾವು ಸರಳ ಆನ್ಮೆಮರಿ ಆದ್ಯತೆ ಸಂಗ್ರಹಣೆಯನ್ನು ಅನುಷ್ಠಾನಗೊಳಿಸುತ್ತೇವೆ (ಉತ್ಪಾದನೆಯಲ್ಲಿ ನೀವು ಇದನ್ನು ಡೇಟಾಬೇಸ್ ಅಥವಾ ವೆಕ್ಟರ್ ಸೂಚ್ಯಂಕದಿಂದ ಬೆಂಬಲಿಸಬೇಕು) ಮತ್ತು ಏಜೆಂಟ್ ಬಳಸಬಹುದಾದ ಉಪಕರಣಗಳಾಗಿ ಇದನ್ನು ಬಹಿರಂಗಪಡಿಸುತ್ತೇವೆ.

### ರಚನೆ
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### ದೃಶ್ಯ 1 — ಮೊದಲ ಬಾರಿ ಬಳಕೆದಾರರು ವಾರ್ಷಿಕೋತ್ಸವ ಯಾತ್ರೆಯನ್ನು ಬುಕ್ ಮಾಡುತ್ತಾರೆ

ಸಾರಾಹ್ ಮೊದಲ ಬಾರಿಗೆ ಭೇಟಿ ನೀಡುತ್ತಾರೆ. ಏಜೆಂಟ್ ಅವಳ ಆದ್ಯತೆಗಳನ್ನು ಸಾಧನಗಳ ಮೂಲಕ ಸಂಗ್ರಹಿಸಿ ಅವುಗಳನ್ನು ಹೊಟೇಲುಗಳನ್ನು ಶಿಫಾರಸು ಮಾಡಲು ಉಪಯೋಗಿಸಬೇಕು.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### ದೃಶ್ಯ 2 — ಸರಾಹ್ ವಾರಗಳ ನಂತರ ಮರುಬರಸುತ್ತಾಳೆ

ಸರಾಹ್ ಒಂದು **ಬೇರೆ ಹೊಸ ತಂತಿ** ಆರಂಭಿಸುತ್ತಾಳೆ (ಹೊಸ ಸೆಷನ್ ಅನ್ನು ಅನುಕರಿಸುತ್ತಾ). ಕೆಲಸ ಮೆಮೋರಿ ಖಾಲಿ ಇದೆ, ಆದರೆ ದೀರ್ಘಕಾಲPreferenceStore ನಲ್ಲಿ ಅವಳ ಮಾಹಿತಿ ಇನ್ನೂ ಇದೆ. ಏಜೆಂಟ್ ಅದನ್ನು ಪಡೆದುಕೊಂಡು ಶಿಫಾರಸುಗಳನ್ನು ವೈಯಕ್ತಿಕಗೊಳಿಸಲು ಬಳಸಬೇಕು.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## ಚುಟುಕು ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೂರು ವಿಧದ ಏಜೆಂಟ್ ಮೆಮೊರಿಯನ್ನು ಮತ್ತು ಅವುಗಳನ್ನು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ ಬಳಸಿ ಹೇಗೆ ಅನುಷ್ಠಾನಗೊಳಿಸುವುದು ಎಂಬುದನ್ನು ಕಲಿತಿರಿ:

| ಮೆಮೊರಿ ಪ್ರಕಾರ | MAF ಮೆಕಾನಿಸಂ | ಆಯುಷ್ಯಕಾಲ |
|---|---|---|
| **ವಾರ್ತಮಾನ** | `agent.create_session()` | ಒಂದೇ ಸಂಭಾಷಣೆ |
| **ಕಲವುಕಾಲಿಕ** | ಥ್ರೆಡ್ ಒಳಗಿನ ಸಂಗ್ರಹಿತ ಸಾಂದರ್ಭಿಕತೆ | ಒಂದೇ ಕಾರ್ಯ / ಸೆಷನ್ |
| **ದೀರ್ಘಕಾಲಿಕ** | `@tool` ಕಾರ್ಯಗಳ ಮೂಲಕ ಪ್ರವೇಶಿಸಬಹುದಾದ ಬಾಹ್ಯ ಸಂಗ್ರಹಣೆ | ಸೆಷನ್‌ಗಳ ನಡುವೆ |

### ಪ್ರಮುಖ ಜ್ಞಾಪಕಗಳು
1. **`agent.create_session()`** ವಾರ್ತಮಾನ ಮೆಮೊರಿಯನ್ನು ಒದಗಿಸುತ್ತದೆ — ಏಜೆಂಟ್ ಒಂದು ಸೆಷನ್ ಒಳಗಿನ ಸಂಪೂರ್ಣ ಸಂಭಾಷಣೆ ಇತಿಹಾಸವನ್ನು ನೋಡುತ್ತದೆ.
2. **ಹೊಸ ಸೆಷನ್‌ಗಳು ಸಾಂದರ್ಭಿಕತೆಯನ್ನು ಕಳೆದುಕೊಳ್ಳುತ್ತವೆ** — ದೀರ್ಘಕಾಲಿಕ ಮೆಮೊರಿ ಇಲ್ಲದೆ ಏಜೆಂಟ್ ಹಿಂದಿನ ಸಂಭಾಷಣೆಗಳನ್ನು ನೆನಪಿಲ್ವ.
3. **`@tool` ಕಾರ್ಯಗಳು**פערವನ್ನು ಸೇತುವೆ ಮಾಡುತ್ತವೆ — ಅವು ಏಜೆಂಟ್‌ಗೆ ಸ್ಥಾಯಿ ಸಂಗ್ರಹಣೆಯಿಂದ ಮಾಹಿತಿಯನ್ನು ಉಳಿಸಲು ಮತ್ತು ಪಡೆದಿಕೊಳ್ಳಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.
4. **ವ್ಯಕ್ತಿಗತೀಕರಣ ಸಮಯದೊಂದಿಗೆ ಉತ್ತಮವಾಗುತ್ತದೆ** — ಹೆಚ್ಚು ಮೆಚ್ಚುಗೆಗಳು ಸಂಗ್ರಹಿಸುವほど ಏಜೆಂಟ್ ನ ಶಿಫಾರಸುಗಳು ಉತ್ತಮವಾಗುತ್ತವೆ.

### ನೈಜ-ಲೋಕ ಅನ್ವಯಿಕೆಗಳು
- **ಗ್ರಾಹಕ ಸೇವೆ**: ಗ್ರಾಹಕರ ಇತಿಹಾಸ ಮತ್ತು ಮೆಚ್ಚುಗೆಗಳನ್ನು ನೆನಪಿಡಿ
- **ವೈಯಕ್ತಿಕ ಸಹಾಯಕರ**: ದಿನಗಳು ಅಥವಾ ವಾರಗಳಷ್ಟು ಸಾಂದರ್ಭಿಕತೆ ಕಾಪಾಡಿಕೊಳ್ಳಿ
- **ಆರೋಗ್ಯಸೇವೆಯು**: ರೋಗಿಯ ಮಾಹಿತಿ ಮತ್ತು ಮೆಚ್ಚುಗೆಗಳನ್ನು ಟ್ರ್ಯಾಕ್ ಮಾಡಿ
- **ಇ-ಕಾಮರ್ಸ್**: ಇತಿಹಾಸ ಆಧಾರಿತ ವೈಯಕ್ತಿಕ ಶಾಪಿಂಗ್

### ಮುಂದಿನ ಹಂತಗಳು
- ಮೆಮೊರಿ ಡಿಕ್ಷನರಿ ಸ್ಥಳವನ್ನು ಡೇಟಾಬೇಸ್ ಅಥವಾ ವೆಕ್ಟರ್ ಸ್ಟೋರ್ (ಉದಾ. ಆಸುರ್ AI ಶೋಧ) ಬಳಸಿ ಬದಲಾಯಿಸಿ
- ಸಮಯ-ಸಂವೇದನಾಶೀಲ ಮಾಹಿತಿಗೆ ಮೆಮೊರಿ ಅವಧಿ ಕೊಡಿ
- ಹಂಚಿಕೆ ಮೆಮೊರಿಯೊಂದಿಗೆ ಬಹು-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳನ್ನು ನಿರ್ಮಿಸಿ
- ಜ್ಞಾನಗ್ರಾಫ್ ಬೆಂಬಲಿತ ಮೆಮೊರಿಗಾಗಿ Cognee ನೋಟ್ಬುಕ್ ಅನ್ನು ಸಂಶೋಧಿಸಿ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
